In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Wpisz dokładne nazwy swoich plików z pasmem 6 i 5
plik_swir = 'nazwa_pasma_6.tif' 
plik_nir = 'nazwa_pasma_5.tif'

with rasterio.open(plik_swir) as src_swir:
    swir = src_swir.read(1).astype('float32')
    transform = src_swir.transform
    crs = src_swir.crs
    width = src_swir.width
    height = src_swir.height

with rasterio.open(plik_nir) as src_nir:
    nir = src_nir.read(1).astype('float32')

# Ignorowanie ostrzeżeń o dzieleniu przez zero
np.seterr(divide='ignore', invalid='ignore')

# Wzór NDBI: (SWIR - NIR) / (SWIR + NIR)
ndbi = (swir - nir) / (swir + nir)
ndbi = np.nan_to_num(ndbi) 

# Maska dla poprawnych danych (tam gdzie SWIR nie jest zerem)
mask = (swir != 0)

# Normalizacja tylko dla faktycznych pikseli danych
ndbi_min = np.min(ndbi[mask])
ndbi_max = np.max(ndbi[mask])
ndbi_norm = (ndbi - ndbi_min) / (ndbi_max - ndbi_min)
ndbi_norm = np.clip(ndbi_norm, 0, 1)

# Wybór palety kolorów
cmap = plt.get_cmap('YlOrRd')
colored_image = cmap(ndbi_norm) # Wynik ma kształt (height, width, 4) z kanałem Alpha

# Ustawienie pełnej przezroczystości (Alpha = 0) dla tła poza sceną
colored_image[~mask, 3] = 0.0 

# Konwersja na format 8-bitowy RGBA (0-255)
ndbi_rgba = (colored_image * 255).astype('uint8')
ndbi_rgba_chw = np.transpose(ndbi_rgba, (2, 0, 1)) # Zmiana układu na (C, H, W)

# Profil z 4 kanałami (RGBA) obsługującym przezroczystość
profil = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 4, # 4 kanały: Red, Green, Blue, Alpha
    'dtype': 'uint8',
    'crs': crs,
    'transform': transform,
    'compress': 'lzw'
}

# Zapis pliku
with rasterio.open('Krakow_NDBI_RGB.tif', 'w', **profil) as dst:
    dst.write(ndbi_rgba_chw)

print("Sukces! Wygenerowano poprawiony, przezroczisty plik Krakow_NDBI_RGB.tif")

Sukces! Wygenerowano kolorowy plik Krakow_NDBI_RGB.tif
